In [7]:
import pyspark
from pyspark.sql import SparkSession

In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 20:19:09 WARN Utils: Your hostname, PradyunPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 20:19:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 20:19:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
df_green = spark.read \
    .option("header", "true") \
    .csv('data/raw/yellow/2020/02/yellow_tripdata_2020_02.csv.gz')

In [11]:
df_green.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)



In [12]:
import pandas as pd

In [13]:
from pyspark.sql import types

In [12]:
df_green_pd = pd.read_csv('data/raw/green/2020/01/green_tripdata_2020_01.csv.gz', nrows = 1000)

In [14]:
df_green_pd.dtypes

VendorID                   int64
lpep_pickup_datetime         str
lpep_dropoff_datetime        str
store_and_fwd_flag           str
RatecodeID                 int64
PULocationID               int64
DOLocationID               int64
passenger_count            int64
trip_distance            float64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
ehail_fee                float64
improvement_surcharge    float64
total_amount             float64
payment_type               int64
trip_type                  int64
congestion_surcharge     float64
dtype: object

In [15]:
spark.createDataFrame(df_green_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('lpep_pickup_datetime', StringType(), True), StructField('lpep_dropoff_datetime', StringType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('RatecodeID', LongType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('ehail_fee', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('payment_type', LongType(), True), StructField('trip_type', LongType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [17]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True), 
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True), 
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True), 
    types.StructField("store_and_fwd_flag", types.StringType(), True), 
    types.StructField("RatecodeID", types.IntegerType(), True), 
    types.StructField("PULocationID", types.IntegerType(), True), 
    types.StructField("DOLocationID", types.IntegerType(), True), 
    types.StructField("passenger_count", types.IntegerType(), True), 
    types.StructField("trip_distance", types.DoubleType(), True), 
    types.StructField("fare_amount", types.DoubleType(), True), 
    types.StructField("extra", types.DoubleType(), True), 
    types.StructField("mta_tax", types.DoubleType(), True), 
    types.StructField("tip_amount", types.DoubleType(), True), 
    types.StructField("tolls_amount", types.DoubleType(), True), 
    types.StructField("ehail_fee", types.DoubleType(), True), 
    types.StructField("improvement_surcharge", types.DoubleType(), True), 
    types.StructField("total_amount", types.DoubleType(), True), 
    types.StructField("payment_type", types.IntegerType(), True), 
    types.StructField("trip_type", types.IntegerType(), True), 
    types.StructField("congestion_surcharge", types.DoubleType(), True)
    ])

In [24]:
year = 2021
taxi_type = 'green'
for month in range(1, 13):
    print(f'processing for {year}\{month:02d}')
    input_path = f'data/raw/{taxi_type}/{year}/{month:02d}/'
    output_path = f'data/pq/{taxi_type}/{year}/{month:02d}/'
    df_green = spark.read \
            .option("header", "true") \
            .schema(green_schema) \
            .csv(input_path)

    df_green \
        .repartition(4) \
        .write.parquet(output_path)


processing for 2021\01
processing for 2021\02
processing for 2021\03
processing for 2021\04
processing for 2021\05
processing for 2021\06
processing for 2021\07
processing for 2021\08
processing for 2021\09


26/03/08 20:11:18 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/green/2021/09/.
java.io.FileNotFoundException: File data/raw/green/2021/09 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/sujatha/projects/batching/data/raw/green/2021/09. SQLSTATE: 42K03

In [21]:
import polars as pl

In [22]:
df_pq = pd.read_parquet('data/pq/yellow/2021/08/part-00000-2e42cb7d-fdc1-4732-937f-d8d4188046b3-c000.snappy.parquet')


In [23]:
df_pq.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge


In [14]:
df_yellow_pd = pd.read_csv('data/raw/yellow/2020/01/yellow_tripdata_2020_01.csv.gz', nrows = 1000)

In [15]:
df_yellow_pd.dtypes

VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object

In [16]:
spark.createDataFrame(df_yellow_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [17]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [20]:
year = 2021
taxi_type = 'yellow'
for month in range(1, 13):
    print(f'processing for {year}\{month:02d}')
    input_path = f'data/raw/{taxi_type}/{year}/{month:02d}/'
    output_path = f'data/pq/{taxi_type}/{year}/{month:02d}/'
    df_green = spark.read \
            .option("header", "true") \
            .schema(yellow_schema) \
            .csv(input_path)

    df_green \
        .repartition(4) \
        .write.parquet(output_path)

processing for 2021\01


processing for 2021\02


processing for 2021\03


processing for 2021\04


processing for 2021\05


...

processing for 2021\07


[Stage 55:>                                                         (0 + 1) / 1]

processing for 2021\08
processing for 2021\09


26/03/08 20:27:13 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/yellow/2021/09/.
java.io.FileNotFoundException: File data/raw/yellow/2021/09 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysi

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/sujatha/projects/batching/data/raw/yellow/2021/09. SQLSTATE: 42K03